# HFP Graft Teşhis Paketi (Faz 1: T1-T4) — forward-only, eğitim YOK
RESULTS.md §15'in "neden"ini ayrıştırır. Tek T4 oturumu, ~30-40 dk.
Girdiler (Add Input): **Qwen2.5-1.5B** (Models) + **checkpoint dataseti**
(`hfp_graft_stage1_son.pt` ve `hfp_graft_final.pt` içermeli).
Kriterler HÜCRE İÇİNDE önceden yazılıdır; çıkan sonuç ne olursa olsun RESULTS.md'ye işlenir.

In [ ]:
# --- 1. KURULUM (egitim notebook'u ile ayni desenler) ---
import os, subprocess, sys, glob
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_XET'] = '1'
import torch
assert torch.cuda.is_available(), 'GPU secili degil! (T4 secin)'
DEV = 'cuda'
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.46'], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)

# Model: Kaggle Input'tan (indirme yok)
def _find_local_model():
    for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
        d = os.path.dirname(cfg)
        if glob.glob(f'{d}/*.safetensors'): return d
    return None
LOCAL_MODEL = _find_local_model()
assert LOCAL_MODEL, 'Qwen input eksik: Add Input > Qwen2.5-1.5B (Models) ekleyin.'

from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype=torch.float32).to(DEV)
model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B param (fp32)')

# Checkpointler: recursive ara
CKPTS = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/hfp_graft_*.pt', recursive=True)}
print('Bulunan checkpointler:', sorted(CKPTS) or 'YOK (T3/T4 icin gerekli)')

# WT-2 valid (PPL referansi) — github raw
import urllib.request, math
dst = f'{BASE}/wt2_valid.txt'
if not os.path.exists(dst):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt', dst)
val_ids = tok(open(dst, encoding='utf-8').read(), return_tensors='pt').input_ids[0]

@torch.no_grad()
def ppl(model, ids, seq=1024, n_chunks=8):     # NOT: 8 chunk — §15 zero-shot (2627) ile ayni olcum
    model.eval(); losses = []
    for i in range(n_chunks):
        s = i * seq
        if s + seq > len(ids): break
        x = ids[s:s+seq].unsqueeze(0).to(DEV)
        losses.append(model(x, labels=x).loss.item())
    return math.exp(sum(losses)/len(losses))
print(f'Hazir. Val: {len(val_ids):,} token')

In [ ]:
# --- 2. T1: NEEDLE KONTROL TESTI (saf Qwen, graft YOK) ---
# ONCEDEN YAZILI KRITER: Tam-attention'li Qwen2.5 (32k baglam) needle'i
# L=2048 ve 8192'de BULMALIDIR. BULURSA test kurgusu saglam -> §15 needle
# sonucu gercek olumsuz sonuctur. BULAMAZSA test kurgusu kirik -> §15'in
# needle satiri modele degil teste yazilir, once test onarilir.
import random
from transformers import DynamicCache

@torch.no_grad()
def needle_raw(model, tok, total_tokens=2048, chunk=512, seed=0):
    random.seed(seed)
    secret = random.choice(['blue falcon', 'copper mountain', 'silent river'])
    needle = f' The secret passphrase is {secret}. '
    filler = ' The weather was ordinary and nothing of note happened that day.'
    fill_ids = tok(filler, add_special_tokens=False).input_ids
    text_ids = []
    while len(text_ids) < total_tokens: text_ids.extend(fill_ids)
    ins = len(text_ids) // 8
    needle_ids = tok(needle, add_special_tokens=False).input_ids
    text_ids = text_ids[:ins] + needle_ids + text_ids[ins:total_tokens - len(needle_ids)]
    query_ids = tok(' The secret passphrase is', add_special_tokens=False).input_ids
    ids = torch.tensor(text_ids + query_ids).unsqueeze(0).to(DEV)
    cache = DynamicCache()
    for s in range(0, ids.size(1), chunk):
        out = model(ids[:, s:s+chunk], past_key_values=cache, use_cache=True)
        cache = out.past_key_values
    gen = []
    last = out.logits[:, -1:].argmax(-1)
    for _ in range(6):
        gen.append(last.item())
        out = model(last, past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        last = out.logits[:, -1:].argmax(-1)
    answer = tok.decode(gen)
    hit = secret.split()[0] in answer
    print(f'  TEACHER L={total_tokens}: gizli="{secret}" cevap="{answer.strip()}" -> {"BULDU" if hit else "BULAMADI"}')
    return hit

print('T1 — saf Qwen needle kontrolu:')
t1_results = {L: needle_raw(model, tok, total_tokens=L) for L in [2048, 8192]}
if all(t1_results.values()):
    print('T1 SONUC: test kurgusu SAGLAM -> §15 needle sonucu gecerli (bellek yolu ogrenmemis).')
else:
    print('T1 SONUC: teacher da BULAMADI -> TEST KURGUSU SUPHELI; §15 needle satirina "test dogrulanamadi" notu dusulmeli, once kurgu onarilmali.')
torch.cuda.empty_cache()

In [ ]:
# --- 3. T2: out_gain INIT A/B (egitimsiz graft, zero-shot PPL) ---
# ONCEDEN YAZILI KRITER: out_gain 1.0 -> ~2627 civari cikmali (§15 replikasyonu).
# out_gain 0.1 zero-shot PPL'i belirgin dusuruyorsa (<1000 hedef), kucuk-init
# ile S1'i yeniden kosmak mantikli yatirimdir. Dusurmuyorsa bu hipotez eleniyor.
from hfp.models.grafting import GraftConfig, graft_llama, set_graft_mode, HFPGraftAttention

# [§15h] Hangi soyun finali test edilecekse o mod secilir: 'exp' | 'cubic_flux_chunked'
DECAY_MODE = 'exp'
gcfg = GraftConfig(decay_mode=DECAY_MODE, write_rule='hybrid',
                   key_feature_map='dpfp', rec_block=16)
GRAFT_LAYERS = graft_llama(model, gcfg)
print(f'Graft: {len(GRAFT_LAYERS)} katman')

set_graft_mode(model, 'student')
ppl_gain10 = ppl(model, val_ids)
print(f'T2a  zero-shot PPL (out_gain=1.0): {ppl_gain10:.1f}   (§15 referans: 2627)')

for m in model.modules():
    if isinstance(m, HFPGraftAttention): m.out_gain.data.fill_(0.1)
ppl_gain01 = ppl(model, val_ids)
print(f'T2b  zero-shot PPL (out_gain=0.1): {ppl_gain01:.1f}')

for m in model.modules():
    if isinstance(m, HFPGraftAttention): m.out_gain.data.fill_(0.01)
ppl_gain001 = ppl(model, val_ids)
print(f'T2c  zero-shot PPL (out_gain=0.01): {ppl_gain001:.1f}')

best = min(ppl_gain10, ppl_gain01, ppl_gain001)
print(f'T2 SONUC: en iyi zero-shot {best:.1f}', '-> <1000 SAGLANDI: kucuk-init S1 yatirimina deger.' if best < 1000 else '-> hala >=1000: init tek basina yetmiyor; agirlik transferi/farkli yol dusunulmeli.')
torch.cuda.empty_cache()

In [ ]:
# --- 4. T3: CHECKPOINT OTOPSISI (stage1_son vs final) ---
# Soru: S2, bellek yolunu gercekten kullanmayi ogrenmis mi?
# alpha ~0.13'te kaldiysa memory katkisi kucuk; out_gain/decay dagilimlari
# olu kafa / doygun kanal gosterir. (Sadece okuma; sayilar RESULTS'a islenir.)
import torch as T

def autopsy(name):
    path = CKPTS.get(name)
    if not path:
        print(f'{name}: input yok, atlandi'); return
    sd = T.load(path, map_location='cpu')
    alphas  = T.cat([v.flatten().float() for k, v in sd.items() if 'alpha_logit' in k])
    gains   = T.cat([v.flatten().float() for k, v in sd.items() if 'out_gain' in k])
    decays  = T.cat([v.flatten().float() for k, v in sd.items() if k.endswith('decay') or '.decay' in k])
    a = T.sigmoid(alphas); d = T.sigmoid(decays)
    print(f'--- {name} ({len(sd)} tensor) ---')
    print(f'  alpha  = sigmoid(alpha_logit): ort {a.mean():.3f}  min {a.min():.3f}  max {a.max():.3f}')
    print(f'  out_gain: ort {gains.mean():.3f}  min {gains.min():.3f}  max {gains.max():.3f}  (|gain|<0.05 olan: {(gains.abs()<0.05).float().mean()*100:.0f}%)')
    print(f'  decay  = sigmoid(decay): ort {d.mean():.3f}  <0.5 olan kanal: {(d<0.5).float().mean()*100:.0f}%')

print('T3 — parametre otopsisi:')
autopsy('hfp_graft_stage1_son.pt')
autopsy('hfp_graft_final.pt')
print('Yorum anahtari: final alpha ~0.13te kaldiysa S2 bellek yolunu buyutmemis;')
print('out_gain cok kuculduyse model bellek ciktisini SUSTURARAK PPLyi korumaya calismis olabilir.')

In [ ]:
# --- 5. T4: EGITILMIS MODELDE KISA-NEEDLE + PPL REPLIKASYONU ---
# ONCEDEN YAZILI KRITER: final.pt ile PPL ~15.9 cikmali (replikasyon).
# Kisa needle (256/512, S2'nin gordugu rejim) de MISS ise sorun baglam
# uzunlugu degil: retrieval hic ogrenilmemis -> Faz 2'de recall-verisi sart.
from hfp.models.grafting import enable_streaming, reset_streaming
import random
from transformers import DynamicCache

_mode_tag = ('exp_' if DECAY_MODE == 'exp' else '')
_finals = {k: v for k, v in CKPTS.items() if 'final' in k and (('exp_' in k) == (DECAY_MODE == 'exp'))}
path = next(iter(sorted(_finals.values())), None)
if not path:
    import glob as _g
    print('Input agacinda bulunanlar:')
    for f in sorted(_g.glob('/kaggle/input/**/*', recursive=True))[:60]: print('  ', f)
    raise AssertionError(
        "final checkpoint input'ta yok/yanlis formatta. DIKKAT: Kaggle .zip yukleyince "
        "OTOMATIK ACAR ve .pt kaybolur -> dosyayi .pt UZANTILI yukleyin. EN KOLAY YOL: "
        "Add Input > Your Work > egitim notebook'unun SON (Run 5) versiyonunu ekleyin; "
        "output'undaki hfp_graft_final.pt dogrudan gorunur.")
sd = torch.load(path, map_location=DEV)
# [KIMLIK KONTROLU] Run 1 finali yanlislikla "run5" adiyla dolasti. Icerikten dogrula:
# Run 1 (out_gain init 1.0 egitimi) -> ort ~0.7; Run 2+ (init 0.1) -> ~0.1-0.3.
_g = torch.cat([v.flatten().float() for k, v in sd.items() if 'out_gain' in k]).mean().item()
print(f'[kimlik] out_gain ort: {_g:.3f} ({path})')
assert _g < 0.5, (f'YANLIS CHECKPOINT: out_gain ort {_g:.2f} -> bu Run 1 kokenli final! '
                  'Dogru dosya: egitim notebookunun SON (Run 5) versiyon output\'undaki '
                  'hfp_graft_final.pt (Add Input > Your Work ile ekleyin).')
res = model.load_state_dict(sd, strict=False)
assert not res.unexpected_keys, f'uyumsuz: {res.unexpected_keys[:5]}'
set_graft_mode(model, 'student'); model.eval()
print(f'final.pt yuklendi ({len(sd)} tensor)')

ppl_final = ppl(model, val_ids)
print(f'T4a  final PPL (8 chunk): {ppl_final:.2f}   (§15: 15.88 @24chunk — yakin cikmali)')

@torch.no_grad()
def needle_student(total_tokens, chunk=128, seed=0):
    random.seed(seed)
    secret = random.choice(['blue falcon', 'copper mountain', 'silent river'])
    needle = f' The secret passphrase is {secret}. '
    filler = ' The weather was ordinary and nothing of note happened that day.'
    fill_ids = tok(filler, add_special_tokens=False).input_ids
    text_ids = []
    while len(text_ids) < total_tokens: text_ids.extend(fill_ids)
    ins = len(text_ids) // 8
    nid = tok(needle, add_special_tokens=False).input_ids
    text_ids = text_ids[:ins] + nid + text_ids[ins:total_tokens - len(nid)]
    qid = tok(' The secret passphrase is', add_special_tokens=False).input_ids
    ids = torch.tensor(text_ids + qid).unsqueeze(0).to(DEV)
    enable_streaming(model, True); reset_streaming(model)
    cache = DynamicCache()
    for s in range(0, ids.size(1), chunk):
        out = model(ids[:, s:s+chunk], past_key_values=cache, use_cache=True)
        cache = out.past_key_values
    gen = []
    last = out.logits[:, -1:].argmax(-1)
    for _ in range(6):
        gen.append(last.item())
        out = model(last, past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        last = out.logits[:, -1:].argmax(-1)
    enable_streaming(model, False)
    ans = tok.decode(gen); hit = secret.split()[0] in ans
    print(f'  STUDENT L={total_tokens}: cevap="{ans.strip()}" -> {"BULDU" if hit else "BULAMADI"}')
    return hit

print('T4b — kisa-needle (S2 rejimi):')
t4 = {L: needle_student(L) for L in [256, 512, 2048]}
if not any(t4.values()):
    print('T4 SONUC: kisa mesafede de MISS -> retrieval HIC ogrenilmemis; Faz 2 secimi: S2 verisine sentetik recall karisimi.')
else:
    print('T4 SONUC: kisa mesafede kismi basari -> sorun olcekle ilgili; uzun-baglam S2 / seq artirimi one cikar.')
print('\\nBitti. Dort sonucu (T1-T4) RESULTS.md ve YOL_HARITASI Faz 2 secimine isleyin.')

In [ ]:
# --- 6. T5: NEEDLE GUVENILIRLIK IZGARASI (Run 5 final.pt; forward-only) ---
# §15f anomalisi (@2048 MISS ama @8192/@16384 FOUND) tek-seed tek-pozisyon.
# Izgara: L x yerlestirme-orani x seed -> guvenilirlik haritasi. Manset iddia
# ancak bu haritayla yapilir. Kriter ONCEDEN: hucre basina >=2/3 isabet "guvenilir".
import random
from transformers import DynamicCache
from hfp.models.grafting import set_graft_mode, enable_streaming, reset_streaming

_mode_tag = ('exp_' if DECAY_MODE == 'exp' else '')
_finals = {k: v for k, v in CKPTS.items() if 'final' in k and (('exp_' in k) == (DECAY_MODE == 'exp'))}
path = next(iter(sorted(_finals.values())), None)
if not path:
    import glob as _g
    print('Input agacinda bulunanlar:')
    for f in sorted(_g.glob('/kaggle/input/**/*', recursive=True))[:60]: print('  ', f)
    raise AssertionError(
        "final checkpoint input'ta yok/yanlis formatta. DIKKAT: Kaggle .zip yukleyince "
        "OTOMATIK ACAR ve .pt kaybolur -> dosyayi .pt UZANTILI yukleyin. EN KOLAY YOL: "
        "Add Input > Your Work > egitim notebook'unun SON (Run 5) versiyonunu ekleyin; "
        "output'undaki hfp_graft_final.pt dogrudan gorunur.")
sd = torch.load(path, map_location=DEV)
# [KIMLIK KONTROLU] Run 1 finali yanlislikla "run5" adiyla dolasti. Icerikten dogrula:
# Run 1 (out_gain init 1.0 egitimi) -> ort ~0.7; Run 2+ (init 0.1) -> ~0.1-0.3.
_g = torch.cat([v.flatten().float() for k, v in sd.items() if 'out_gain' in k]).mean().item()
print(f'[kimlik] out_gain ort: {_g:.3f} ({path})')
assert _g < 0.5, (f'YANLIS CHECKPOINT: out_gain ort {_g:.2f} -> bu Run 1 kokenli final! '
                  'Dogru dosya: egitim notebookunun SON (Run 5) versiyon output\'undaki '
                  'hfp_graft_final.pt (Add Input > Your Work ile ekleyin).')
res = model.load_state_dict(sd, strict=False)
assert not res.unexpected_keys
set_graft_mode(model, 'student'); model.eval()
print(f'Run5 final.pt yuklendi ({len(sd)} tensor)')

# Egitimde HIC gecmeyen kelimeler + tam-ifade kriteri (cell 7 ile ayni sertlik)
_SECRETS = ['orange kettle', 'purple ladder', 'crimson garden']

@torch.no_grad()
def needle_at(total_tokens, ins_frac, seed, chunk=128):
    random.seed(seed)
    secret = random.choice(_SECRETS)
    needle = f' The secret passphrase is {secret}. '
    filler = ' The weather was ordinary and nothing of note happened that day.'
    fill_ids = tok(filler, add_special_tokens=False).input_ids
    text_ids = []
    while len(text_ids) < total_tokens: text_ids.extend(fill_ids)
    ins = int(total_tokens * ins_frac)
    nid = tok(needle, add_special_tokens=False).input_ids
    text_ids = text_ids[:ins] + nid + text_ids[ins:total_tokens - len(nid)]
    qid = tok(' The secret passphrase is', add_special_tokens=False).input_ids
    ids = torch.tensor(text_ids + qid).unsqueeze(0).to(DEV)
    enable_streaming(model, True); reset_streaming(model)
    cache = DynamicCache()
    for s in range(0, ids.size(1), chunk):
        out = model(ids[:, s:s+chunk], past_key_values=cache, use_cache=True)
        cache = out.past_key_values
    gen = []
    last = out.logits[:, -1:].argmax(-1)
    for _ in range(6):
        gen.append(last.item())
        out = model(last, past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        last = out.logits[:, -1:].argmax(-1)
    enable_streaming(model, False)
    return secret in tok.decode(gen)

print('T5 izgara (L x yerlestirme x 3 seed):')
print(f'{"L":>6} {"konum":>6} | isabet')
grid = {}
for L in [512, 1024, 2048, 4096, 8192]:
    for frac in [0.125, 0.5, 0.875]:
        hits = sum(needle_at(L, frac, sd_) for sd_ in [0, 1, 2])
        grid[(L, frac)] = hits
        tag = 'GUVENILIR' if hits >= 2 else ('KISMEN' if hits == 1 else 'YOK')
        print(f'{L:>6} {frac:>6} | {hits}/3  {tag}', flush=True)
total = sum(grid.values()); n = len(grid) * 3
print(f'\nT5 SONUC: toplam {total}/{n} isabet. Harita RESULTS §15g\'ye islenecek —')
print('duzenli bir mesafe/konum deseni var mi, @2048 anomalisi tekrarlaniyor mu, ona bak.')
